# Joint Distributions and Extra Axes

By default a parametric population is the product of a star formation history and a metallicity distribution: every age has the same distribution of metallicities. This example covers how to go beyond that, with distributions that depend on the age of the stars (e.g. an enrichment history), and with bins along axes other than age and metallicity (e.g. the ionisation parameter of a nebular grid).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from unyt import Gyr, Msun

from synthesizer.emission_models import IncidentEmission
from synthesizer.grid import Grid
from synthesizer.parametric import SFH, Stars, ZDist

grid = Grid("test_grid")
model = IncidentEmission(grid)

## Age dependent metallicity distributions

Any parameter of a metallicity distribution (`ZDist`) can be a function of the stars' age. It is called with the age (a `unyt_quantity` or `unyt_array`) and should return the parameter's value. Here an enrichment history, where the oldest stars are the most metal poor:

In [ ]:
def z_of_age(age):
    """Return the metallicity of stars formed age ago."""
    return 0.02 * np.exp(-age.to("Gyr").value / 3)


enriched = Stars(
    grid.log10ages,
    grid.metallicities,
    sf_hist=SFH.Constant(max_age=10 * Gyr),
    metal_dist=ZDist.DeltaConstant(metallicity=z_of_age),
    initial_mass=1e10 * Msun,
)
enriched.plot_sfzh(show=True)

The distribution is averaged over the mass formed within each age bin (weighting by the star formation history), so a bin over which the metallicity changes holds the range of metallicities formed in it.

Distributions with a spread can depend on age in the same way, e.g. a log-normal distribution (`ZDist.LogNormal`, normal in log10 Z) whose median follows the enrichment history:

In [ ]:
spread = Stars(
    grid.log10ages,
    grid.metallicities,
    sf_hist=SFH.Constant(max_age=10 * Gyr),
    metal_dist=ZDist.LogNormal(
        log10metallicity=lambda age: np.log10(z_of_age(age)), sigma=0.2
    ),
    initial_mass=1e10 * Msun,
)
spread.plot_sfzh(show=True)

In [ ]:
fig, ax = plt.subplots()
for stars, label in ((enriched, "DeltaConstant"), (spread, "LogNormal")):
    sed = stars.get_spectra(model)
    ax.loglog(sed.lam, sed.lnu, label=label)
ax.set_xlim(1e3, 1e5)
ax.set_ylim(1e26, None)
ax.set_xlabel(r"$\lambda / \AA$")
ax.set_ylabel(r"$L_\nu / \mathrm{erg\,s^{-1}\,Hz^{-1}}$")
ax.legend()
plt.show()

## Extra axes

Grids can have axes beyond age and metallicity, e.g. the ionisation parameter or hydrogen density of the photoionisation modelling. A parametric `Stars` is matched to a grid's axes by name when the emission is extracted:

- An axis the `Stars` has bins along uses those bins.
- A grid axis the `Stars` has no bins along takes its value exactly as particle attributes do: from the emission model's fixed parameters, or else an attribute of the `Stars` (one value for every population, or one per population). That value is a zero width bin.
- An axis the `Stars` has bins along but the grid doesn't is summed over.

So for the common case of a single value nothing is needed beyond what particles need, e.g. `IncidentEmission(grid, ionisation_parameter=0.01)` or `Stars(..., ionisation_parameter=0.01)`.

To give a distribution along an extra axis use `extra_axes`, which combines each distribution with the age and metallicity bins. Each entry is a single value, or a tuple of bin edges and either the weight in each bin or a distribution object (anything with a `get_bin_weights(edges)` method, e.g. a `ZDist`):

In [ ]:
stars = Stars(
    grid.log10ages,
    grid.metallicities,
    sf_hist=SFH.Constant(max_age=1 * Gyr),
    metal_dist=0.01,
    initial_mass=1e9 * Msun,
    extra_axes={
        # Three ionisation parameters, weighted 1:2:1 (zero width bins at
        # the three values, with empty bins between them)
        "ionisation_parameter": (
            [1e-3, 1e-3, 1e-2, 1e-2, 1e-1, 1e-1],
            [1.0, 0.0, 2.0, 0.0, 1.0],
        ),
    },
)
print(stars.bin_axes)
print(stars.bin_masses.shape)

The test grid has no ionisation parameter axis, so here the extra axis is summed over and the emission is unchanged. On a grid with that axis each population's mass would be split between the three values.

In [ ]:
plain = Stars(
    grid.log10ages,
    grid.metallicities,
    sf_hist=SFH.Constant(max_age=1 * Gyr),
    metal_dist=0.01,
    initial_mass=1e9 * Msun,
)
print(np.allclose(stars.get_spectra(model).lnu, plain.get_spectra(model).lnu))

A value on an extra axis can also depend on age, given as the points to place it on and a function of age (here an ionisation parameter falling as the stars age):

In [ ]:
points = np.logspace(-4, -1, 7)
aging = Stars(
    grid.log10ages,
    grid.metallicities,
    sf_hist=SFH.Constant(max_age=1 * Gyr),
    metal_dist=0.01,
    initial_mass=1e9 * Msun,
    extra_axes={
        "ionisation_parameter": (
            points,
            lambda age: 0.1 * (1 + age.to("Myr").value) ** -1,
        ),
    },
)
print(aging.bin_masses.shape)

Finally, a full joint distribution can be passed to `Stars.from_binned` with `extra_edges`, giving the edges along each extra axis in the order of the masses' trailing axes:

In [ ]:
masses = np.random.default_rng(0).random((3, 1, 2)) * 1e9
joint = Stars.from_binned(
    grid.log10ages,
    grid.metallicities,
    [0.0, 1e7, 1e8, 1e9],  # age bins (yr)
    [0.01, 0.01],  # a single metallicity
    masses * Msun,
    extra_edges={"ionisation_parameter": [1e-3, 1e-2, 1e-1]},
)
print(joint.bin_axes, joint.bin_masses.shape)